# Chapter 9: Model Evaluation & Hyperparameter Tuning

How do we know if our model is actually good? In this unit, we explore evaluation strategies and optimization techniques.

### High-Level Intuition & Analogy
* **Cross-Validation** is like giving a student multiple practice exams. Instead of testing on a single split, we partition data into $K$ blocks. We train on $K-1$ blocks and test on the remaining block, repeating this $K$ times so every data point is tested once.
* **Hyperparameter Tuning** is like dialing in the settings of an engine. We don't tune weights (parameters); we tune parameters that control the model structure (hyperparameters), like tree depth.

### Core Concepts & Step-by-Step Execution
1. **K-Fold CV Execution**:
   * Split the dataset into $K$ equal-sized folds.
   * For each fold: train the model on the other $K-1$ folds, evaluate it on this fold, and record the score.
   * Average the scores to get the final validation result.
2. **Grid Search vs. Random Search**:
   * **Grid Search** checks every possible combination of settings from a list. It is thorough but slow.
   * **Random Search** randomly picks settings from ranges. It is fast and often finds optimal configurations in a fraction of the time.
3. **Classification Threshold Tuning**: In binary classification, changing the probability threshold (default 0.5) lets you balance precision (avoiding false alarms) and recall (avoiding missed detections).

### Mathematical Foundations (Simplified)
* **Precision**: True Positives divided by all predicted positives. Formula: $P = \frac{TP}{TP + FP}$.
* **Recall**: True Positives divided by all actual positives. Formula: $R = \frac{TP}{TP + FN}$.
* **F1-Score**: Harmonic mean of precision and recall. Formula: $F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$.

### Pros and Cons
* **Pros**: Prevents validation bias, and automatically optimizes model settings.
* **Cons**: Extremely computationally expensive, and grid search can miss peak values between grid steps.

### Pro-Tips & Gotchas
* **Leakage in CV**: Ensure all preprocessing steps (scaling, encoding) are executed *within* the cross-validation loop (using Scikit-Learn Pipelines). If you preprocess the entire dataset before running cross-validation, your evaluation scores will be optimistically biased.

### Programming Guide & Key Functions
Here is a complete setup code containing search models, metrics, and threshold curves:

```python
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_curve, roc_auc_score
import numpy as np

X = np.random.rand(100, 5)
y = np.random.randint(0, 2, 100)

# --- Detailed Cross Validation ---
model = RandomForestClassifier(random_state=42)
cv_results = cross_validate(model, X, y, cv=5, scoring=['accuracy', 'f1_macro'], return_train_score=False)
print("Mean Accuracy:", np.mean(cv_results['test_accuracy']))

# --- Hyperparameter Randomized Search ---
param_dist = {
    'n_estimators': [10, 50, 100],
    'max_depth': [3, 5, None]
}
rand_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=5, cv=3, random_state=42)
rand_search.fit(X, y)
print("Best Params:", rand_search.best_params_)

# --- Evaluation Metrics ---
y_test = [0, 1, 0, 1, 1, 0]
y_prob = [0.1, 0.9, 0.3, 0.4, 0.8, 0.2]
y_pred = [0, 1, 0, 0, 1, 0]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Precision-Recall & ROC Curves
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_prob)
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
```

In [ ]:
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=100, random_state=42)
rf = RandomForestClassifier(n_estimators=10, random_state=42)
scores = cross_val_score(rf, X, y, cv=5)
print("CV Scores:", scores)
print("Mean CV Score:", scores.mean())

---
## Exercises

Now it is your turn! Run the verification code cell after each exercise to check your answers.

### Exercise 1: Train-Test Split (Easy)
Split features `X` and labels `y` into training and testing sets (80% train, 20% test). Store outputs in `X_train`, `X_test`, `y_train`, `y_test`. Set `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

X = np.arange(10).reshape((5, 2))
y = np.array([0, 1, 0, 1, 0])
# TODO: Train-test split (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = ____
print(X_train.shape)

In [ ]:
from learntools import ch9

ch9.step_1.check(X_train, X_test, y_train, y_test)
# ch9.step_1.hint()# ch9.step_1.solution()

### Exercise 2: KFold Instantiation (Easy)
Import `KFold` from `sklearn.model_selection` and instantiate it with `n_splits=5`, `shuffle=True`, and `random_state=42`. Store it in `kf`.

In [ ]:
# TODO: Import and instantiate KFold with 5 splits
kf = ____
print(kf)

In [ ]:
ch9.step_2.check(kf)
# ch9.step_2.hint()# ch9.step_2.solution()

### Exercise 3: Calculate Precision Score (Easy)
Calculate the classification precision score of `y_pred` against `y_true`. Store it in `precision`.

In [ ]:
from sklearn.metrics import precision_score

y_true = np.array([0, 1, 1, 0])
y_pred = np.array([0, 1, 0, 0])
# TODO: Calculate precision score
precision = ____
print(precision)

In [ ]:
ch9.step_3.check(precision)
# ch9.step_3.hint()# ch9.step_3.solution()

### Exercise 4: Confusion Matrix (Easy)

Calculate the confusion matrix between the ground truth labels `y_true_cm` and model predictions `y_pred_cm` using `sklearn.metrics.confusion_matrix` and store it in `cm`.

In [ ]:
from sklearn.metrics import confusion_matrix

y_true_cm = np.array([0, 1, 0, 1, 1])

y_pred_cm = np.array([0, 1, 1, 0, 1])

# TODO: Calculate confusion matrix

cm = ____

print(cm)

In [ ]:
ch9.step_7.check(cm)

# ch9.step_7.hint()# ch9.step_7.solution()

### Exercise 5: Precision Score (Easy/Medium)

Calculate the precision score of the predictions `y_pred_cm` relative to `y_true_cm` using `sklearn.metrics.precision_score` and store it in `precision_val`.

In [ ]:
from sklearn.metrics import precision_score

# TODO: Compute precision score

precision_val = ____

print("Precision:", precision_val)

In [ ]:
ch9.step_8.check(precision_val)

# ch9.step_8.hint()# ch9.step_8.solution()

### Exercise 6: K-Fold Cross-Validation (Medium)
Perform a 5-split K-Fold Cross-Validation (`KFold` with `n_splits=5`, `shuffle=True`, and `random_state=42`) using `cross_val_score` on the provided `RandomForestClassifier` (`rf`) and datasets.
Compute and store the mean of the cross-validation scores in `mean_cv_score`.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

# TODO: Configure KFold and compute mean_cv_score
mean_cv_score = ____

print("Mean CV Accuracy Score:", mean_cv_score)

In [ ]:
ch9.step_4.check(mean_cv_score)
# ch9.step_4.hint()# ch9.step_4.solution()

### Exercise 7: Grid Search vs. Randomized Search Tuning (Medium)
Tune the parameters of a `DecisionTreeRegressor` using Grid Search and Random Search.
Search space:
- `'max_depth'`: `[3, 5, 7]`
- `'min_samples_split'`: `[2, 5, 10]`

1. Fit `GridSearchCV` on the dataset `X, y`.
2. Fit `RandomizedSearchCV` on the dataset `X, y` with `n_iter=5` and `random_state=42`.
Store the dictionaries of the best parameters found in variables named `best_params_grid` and `best_params_random` respectively.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=500, n_features=5, noise=0.1, random_state=42)

# TODO: Fit Grid Search and Random Search. Extract best parameters.
best_params_grid = ____
best_params_random = ____

print("Grid Search Best Parameters:", best_params_grid)
print("Random Search Best Parameters:", best_params_random)

In [ ]:
ch9.step_5.check(best_params_grid, best_params_random)
# ch9.step_5.hint()# ch9.step_5.solution()

### Exercise 8: F1-Score Classification Threshold Tuning (Hard)
By default, classifiers predict class 1 if the probability is $\ge 0.5$. In imbalanced sets, this threshold might not be optimal.
Given predictions probabilities `y_probs` and true targets `y_test`:
Find the classification threshold in the range $[0.1, 0.9]$ (inclusive, with steps of `0.01`) that maximizes the $F_1$ score.
Store:
1. The optimal threshold value in `best_threshold`.
2. The maximum F1 score in `max_f1`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=500, n_classes=2, weights=[0.7, 0.3], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
clf = LogisticRegression(random_state=42).fit(X_train, y_train)
y_probs = clf.predict_proba(X_test)[:, 1]

# TODO: Find threshold in np.arange(0.1, 0.91, 0.01) maximizing F1-score
best_threshold = 0.0
max_f1 = 0.0

print(f"Optimal Threshold: {best_threshold:.2f} with Max F1: {max_f1:.4f}")

In [ ]:
ch9.step_6.check(best_threshold, max_f1)
# ch9.step_6.hint()# ch9.step_6.solution()